本 notebook 是 [M05.2 自动微分](../notes/02-automatic-differentiation.qmd) 的实例讲解。

1. 1. 标量计算图、共享路径与小 MLP
2. 2. JVP、VJP 与对偶一致性
3. 3. LJ 能量、力与固定分数坐标应力
4. 4. create_graph 与力损失的参数梯度
5. 5. 用同一 autograd 力运行 NVE

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

运行前安装已声明依赖：`uv sync --locked --extra perf --extra data --extra dl`。本例使用 CPU，不需外部数据或预训练权重。

In [1]:
import math

import numpy as np
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from torch.func import jacrev, jvp, vjp

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. 标量计算图、共享路径与小 MLP

每个节点保存数值和父边局部导数，反向按拓扑逆序累加；用共享变量核验，再用这一个引擎训练 XOR 小网络。

In [2]:
torch.set_num_threads(1)


class Scalar:
    """教学用无量纲标量反向模式，只支持一阶导数。"""

    def __init__(self, data, parents=()):
        self.data = float(data)
        self.parents = parents
        self.grad = 0.0

    @staticmethod
    def wrap(value):
        return value if isinstance(value, Scalar) else Scalar(value)

    def __add__(self, other):
        other = self.wrap(other)
        return Scalar(self.data + other.data, ((self, 1.0), (other, 1.0)))

    __radd__ = __add__

    def __mul__(self, other):
        other = self.wrap(other)
        return Scalar(self.data * other.data, ((self, other.data), (other, self.data)))

    __rmul__ = __mul__

    def __neg__(self):
        return self * (-1.0)

    def __sub__(self, other):
        return self + (-self.wrap(other))

    def __pow__(self, power):
        return Scalar(self.data**power, ((self, power * self.data ** (power - 1)),))

    def tanh(self):
        value = math.tanh(self.data)
        return Scalar(value, ((self, 1 - value * value),))

    def backward(self):
        order = []
        seen = set()

        def visit(node):
            if node in seen:
                return
            seen.add(node)
            for parent, _ in node.parents:
                visit(parent)
            order.append(node)

        visit(self)
        for node in order:
            node.grad = 0.0
        self.grad = 1.0
        for node in reversed(order):
            for parent, partial in node.parents:
                parent.grad += node.grad * partial


a = Scalar(2.0)
z = a * a + a
z.backward()
print("共享路径导数:", a.grad)
assert a.grad == 5.0
local = np.random.default_rng(42)
hidden = 6
w1 = [[Scalar(v) for v in local.normal(0, 0.6, 2)] for _ in range(hidden)]
b1 = [Scalar(0) for _ in range(hidden)]
w2 = [Scalar(v) for v in local.normal(0, 0.6, hidden)]
b2 = Scalar(0)
parameters = [p for row in w1 for p in row] + b1 + w2 + [b2]


def predict(values):
    h = [
        (sum(w * x for w, x in zip(row, values, strict=True)) + bias).tanh()
        for row, bias in zip(w1, b1, strict=True)
    ]
    return sum(w * x for w, x in zip(w2, h, strict=True)) + b2


X = [[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]]
target = [-1.0, 1.0, 1.0, -1.0]
history = []
for _ in range(900):
    loss = sum((predict(x) - y) ** 2 for x, y in zip(X, target, strict=True)) * 0.25
    loss.backward()
    history.append(loss.data)
    for p in parameters:
        p.data -= 0.08 * p.grad
print("首末 MSE:", history[0], history[-1], "XOR 预测:", [predict(x).data for x in X])
assert history[-1] < 0.01

共享路径导数: 5.0
首末 MSE: 1.0560365747696396 1.217187724852733e-30 XOR 预测: [-0.9999999999999987, 0.9999999999999992, 0.999999999999999, -0.9999999999999988]


读法：共享参数路径贡献必须累加而不是覆盖；该引擎每次 backward 清空可达节点梯度，与 PyTorch 默认累加语义不同。只保存数值局部导数，因而不支持高阶图。

## 2. JVP、VJP 与对偶一致性

用 torch.func 的变换计算方向导数与余切传播，与显式 Jacobian 对齐。

In [3]:
def vector_function(x):
    return torch.stack([x[0] * x[1], torch.sin(x[0]), x[1] ** 2])


x = torch.tensor([0.7, -1.2], dtype=torch.float64)
v = torch.tensor([0.2, 0.4], dtype=torch.float64)
u = torch.tensor([1.0, -2.0, 0.5], dtype=torch.float64)
y, jv = jvp(vector_function, (x,), (v,))
_, pullback = vjp(vector_function, x)
jtu = pullback(u)[0]
J = jacrev(vector_function)(x)
print("J:\n", J, "\nJv:", jv, "Jᵀu:", jtu)
print("对偶残差:", torch.dot(u, jv) - torch.dot(jtu, v))
assert torch.allclose(jv, J @ v, atol=1e-12)
assert torch.allclose(jtu, J.T @ u, atol=1e-12)

/Users/qdxu/Downloads/learningmatter/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


J:
 tensor([[-1.2000,  0.7000],
        [ 0.7648,  0.0000],
        [ 0.0000, -2.4000]], dtype=torch.float64) 
Jv: tensor([ 0.0400,  0.1530, -0.9600], dtype=torch.float64) Jᵀu: tensor([-2.7297, -0.5000], dtype=torch.float64)
对偶残差: tensor(0., dtype=torch.float64)


读法：前向模式沿输入方向传播 Jv，反向模式沿输出权重传播 Jᵀu；标量损失对许多参数求梯度适合反向模式，不必显式生成完整 Jacobian。

## 3. LJ 能量、力与固定分数坐标应力

用双精度自动微分计算三个粒子的非周期 LJ 体系；力与应变导数分别用中心差分核验，明确应力符号与体积约定。

In [4]:
dtype = torch.float64


def lj(positions):
    i, j = torch.triu_indices(len(positions), len(positions), offset=1)
    distance = torch.linalg.vector_norm(positions[i] - positions[j], dim=1)
    inv6 = distance.pow(-6)
    return (4 * (inv6**2 - inv6)).sum()


positions = torch.tensor(
    [[0.0, 0.0, 0.0], [1.3, 0.0, 0.0], [0.2, 1.5, 0.1]], dtype=dtype, requires_grad=True
)
E = lj(positions)
force = -torch.autograd.grad(E, positions)[0]
h = 1e-5
fd = torch.zeros_like(positions)
for i in range(len(positions)):
    for j in range(3):
        shift = torch.zeros_like(positions)
        shift[i, j] = h
        fd[i, j] = -(lj(positions.detach() + shift) - lj(positions.detach() - shift)) / (2 * h)
relative = torch.linalg.vector_norm(force - fd) / torch.linalg.vector_norm(fd)
strain = torch.zeros((3, 3), dtype=dtype, requires_grad=True)
volume = 64.0
deformed = positions.detach() @ (torch.eye(3, dtype=dtype) + strain).T
stress = torch.autograd.grad(lj(deformed), strain)[0] / volume
stress_fd = torch.zeros_like(strain)
for a in range(3):
    for b in range(3):
        shift = torch.zeros_like(strain)
        shift[a, b] = h
        plus = positions.detach() @ (torch.eye(3, dtype=dtype) + shift).T
        minus = positions.detach() @ (torch.eye(3, dtype=dtype) - shift).T
        stress_fd[a, b] = (lj(plus) - lj(minus)) / (2 * h * volume)
stress_relative = torch.linalg.vector_norm(stress - stress_fd) / torch.linalg.vector_norm(stress_fd)
print("force:\n", force.detach(), "净力:", force.sum(dim=0), "相对差分误差:", relative.item())
print("stress:\n", stress, "应力相对差分误差:", stress_relative.item())
assert relative < 1e-6 and stress_relative < 1e-6
assert torch.linalg.vector_norm(force.sum(dim=0)) < 1e-12

force:
 tensor([[ 2.3833,  1.0750,  0.0717],
        [-2.4134,  0.2364,  0.0158],
        [ 0.0300, -1.3114, -0.0874]], dtype=torch.float64) 净力: tensor([5.5511e-17, 0.0000e+00, 0.0000e+00], dtype=torch.float64) 相对差分误差: 2.8847682612200044e-10
stress:
 tensor([[ 4.8927e-02, -7.0413e-04, -4.6942e-05],
        [-7.0413e-04,  3.0736e-02,  2.0491e-03],
        [-4.6942e-05,  2.0491e-03,  1.3660e-04]], dtype=torch.float64) 应力相对差分误差: 4.887902935384066e-10


读法：这里单位采用 ε=σ=m=1，固定分数坐标时位置随仿射应变移动；取应力为正的能量应变导数除以参考体积，不含动能，也没有周期边界、截断或近邻列表。

## 4. create_graph 与力损失的参数梯度

若损失依赖力，必须保留输入梯度的可微计算图；用可解析幅度参数检查混合二阶导数。

In [5]:
q = torch.tensor([0.7, 1.3], dtype=torch.float64, requires_grad=True)
a = torch.tensor(1.2, dtype=torch.float64, requires_grad=True)
energy = 0.5 * a * (q * q).sum()
force = -torch.autograd.grad(energy, q, create_graph=True)[0]
target = torch.tensor([-0.5, -1.0], dtype=torch.float64)
loss = ((force - target) ** 2).mean()
grad = torch.autograd.grad(loss, a)[0]
analytic = 2 * ((-a.detach() * q.detach() - target) * (-q.detach())).mean()
print("力损失参数梯度/解析:", grad.item(), analytic.item())
assert torch.allclose(grad, analytic, atol=1e-12)
q2 = q.detach().clone().requires_grad_()
a2 = a.detach().clone().requires_grad_()
force_without = -torch.autograd.grad(0.5 * a2 * (q2 * q2).sum(), q2)[0]
print("未 create_graph 的力是否可继续求导:", force_without.requires_grad)

力损失参数梯度/解析: 0.9660000000000001 0.9660000000000001
未 create_graph 的力是否可继续求导: False


读法：retain_graph 保留已有图供重复反向，create_graph 才为导数运算构建可微图；detach、numpy 转换和错误的 no_grad 会切断参数梯度路径。

## 5. 用同一 autograd 力运行 NVE

对 LJ 二聚体用 velocity Verlet，比较两种步长在固定总时长内的最大相对能量偏差。每一步重建坐标叶节点并释放图。

In [6]:
def energy_force(q):
    leaf = q.detach().clone().requires_grad_()
    energy = lj(leaf)
    force = -torch.autograd.grad(energy, leaf)[0]
    return energy.detach(), force.detach()


def simulate(dt, total=2.0):
    q = torch.tensor([[-0.65, 0.0, 0.0], [0.65, 0.0, 0.0]], dtype=torch.float64)
    velocity = torch.tensor([[0.0, 0.06, 0.0], [0.0, -0.06, 0.0]], dtype=torch.float64)
    e, force = energy_force(q)
    initial = (e + 0.5 * (velocity**2).sum()).item()
    history = []
    for _ in range(round(total / dt)):
        velocity += 0.5 * dt * force
        q += dt * velocity
        e, force = energy_force(q)
        velocity += 0.5 * dt * force
        history.append((e + 0.5 * (velocity**2).sum()).item())
    return np.array(history), initial


records = []
for dt in [0.004, 0.002]:
    values, initial = simulate(dt)
    deviation = np.max(np.abs(values - initial)) / abs(initial)
    records.append(deviation)
    print("dt", dt, "最大相对能量偏差", deviation)
assert records[1] < records[0]

dt 0.004 最大相对能量偏差 0.00025582695684922476
dt 0.002 最大相对能量偏差 6.398497544519276e-05


读法：有限步长 Verlet 不精确守恒能量；缩步后偏差下降是积分一致性的证据。每一步 detach 避免把整条轨迹无意连接成巨大反向图，长程稳定性仍需更广验证。

## 小结

AD 是对计算依赖的链式法则组织。拓扑逆序与梯度累加解释反向引擎，JVP/VJP 解释传播方向，高阶图解释力训练。把能量、坐标、应变和单位约定写清，再用差分、对称性与积分收敛交叉核验，才能可靠地将自动微分接入科学模型。